In [ ]:
# Cell 1: Imports and MongoDB connection via CRUD module
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, dash_table
from crud import AnimalShelter

# Connect to MongoDB through the CRUD module.
# The AnimalShelter constructor creates performance indexes automatically
# on animal_type, breed, sex_upon_outcome, age_upon_outcome_in_weeks,
# and outcome_type the first time it runs.
shelter = AnimalShelter(
    user='aacuser',
    passwd='SNHU1234',
    host='nv-desktop-services.apporto.com',
    port=30091,
    db='AAC',
    collection='animals'
)

# Load the full collection into a DataFrame as the baseline dataset.
# The read() call uses the indexed collection, so the query is fast
# even at 10 000+ documents.
df = pd.DataFrame(shelter.read({}, projection={'_id': 0}))
print(f"Loaded {len(df)} records from MongoDB.")
df.head()


In [ ]:
# Cell 2: Rescue filter definitions
#
# Each filter is expressed as a MongoDB query dict so it can be passed
# directly to shelter.read() and shelter.aggregate_outcomes(), keeping
# the filtering logic in the database layer rather than in Python.
#
# Criteria are drawn from the Grazioso Salvare rescue-training
# specifications used in the original CS 340 project:
#   - Water Rescue: Labrador Retriever Mix, Chesapeake Bay Retriever,
#     Newfoundland; intact female; 26-156 weeks
#   - Mountain/Wilderness: German Shepherd, Alaskan Malamute,
#     Old English Sheepdog, Siberian Husky, Rottweiler;
#     intact male; 26-156 weeks
#   - Disaster/Tracking: Doberman Pinscher, German Shepherd,
#     Golden Retriever, Bloodhound, Rottweiler;
#     intact male; 20-300 weeks

def build_filter_map():
    """Return a dict mapping filter label to a MongoDB query dict."""
    return {
        "Reset": {},
        "Water Rescue": {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland",
                ]
            },
        },
        "Mountain or Wilderness Rescue": {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "breed": {
                "$in": [
                    "German Shepherd Mix",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler",
                ]
            },
        },
        "Disaster Rescue or Individual Tracking": {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd Mix",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler",
                ]
            },
        },
    }

filter_map = build_filter_map()
print("Filter keys:", list(filter_map.keys()))


In [ ]:
# Cell 3: Initialise the Dash application
app = Dash(__name__)


In [ ]:
# Cell 4: Dashboard layout
app.layout = html.Div([
    html.H1("Grazioso Salvare Animal Shelter Dashboard"),

    # Branding
    html.Div([
        html.A(
            html.Img(
                src="https://upload.wikimedia.org/wikipedia/commons/1/1f/SNHU_Logo.png",
                height="50px"
            ),
            href="https://www.snhu.edu"
        ),
        html.Span(
            "Dashboard by Ethan Chapman",
            style={"margin-left": "20px", "font-weight": "bold"}
        )
    ], style={"display": "flex", "align-items": "center", "margin-bottom": "20px"}),

    # Filter controls
    html.Div([
        html.Label("Select Rescue Filter:"),
        dcc.Dropdown(
            id='filter-dropdown',
            options=[{"label": k, "value": k} for k in filter_map.keys()],
            value="Reset",
            clearable=False
        ),
        html.Br(),
        html.Label("Filter by Age (weeks):"),
        dcc.RangeSlider(
            id='age-slider',
            min=0,
            max=int(df['age_upon_outcome_in_weeks'].max()),
            step=1,
            value=[0, int(df['age_upon_outcome_in_weeks'].max())],
            marks={
                0: '0',
                int(df['age_upon_outcome_in_weeks'].max()): str(int(df['age_upon_outcome_in_weeks'].max()))
            },
            tooltip={"placement": "bottom", "always_visible": True}
        )
    ], style={"width": "60%", "margin-bottom": "20px"}),

    # Data table
    dash_table.DataTable(
        id='data-table',
        columns=[{"name": i, "id": i} for i in df.columns],
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'}
    ),

    html.Br(),

    # Charts row
    html.Div([
        dcc.Graph(id='outcome-pie-chart'),
        dcc.Graph(id='breed-bar-chart'),
        dcc.Graph(id='geo-map')
    ])
])


In [ ]:
# Cell 5: Callbacks
#
# The callback runs two distinct database operations:
#   1. shelter.read() retrieves the documents needed for the data table
#      and the geolocation map, applying the rescue query and the age
#      range together so only matching records travel over the network.
#   2. shelter.aggregate_outcomes() runs a MongoDB aggregation pipeline
#      for the pie chart, which is more efficient than grouping in Pandas
#      because the counting happens inside the database engine.

@app.callback(
    Output('data-table', 'data'),
    Output('outcome-pie-chart', 'figure'),
    Output('breed-bar-chart', 'figure'),
    Output('geo-map', 'figure'),
    Input('filter-dropdown', 'value'),
    Input('age-slider', 'value')
)
def update_dashboard(selected_filter, age_range):
    # Build the compound query by combining the rescue filter with the
    # age range constraint.
    base_query = filter_map[selected_filter].copy()

    # Merge the age slider range into the query.
    # If the filter already restricts age, intersect rather than overwrite.
    age_constraint = {"$gte": age_range[0], "$lte": age_range[1]}
    if "age_upon_outcome_in_weeks" in base_query:
        existing = base_query["age_upon_outcome_in_weeks"]
        age_constraint["$gte"] = max(existing.get("$gte", 0), age_range[0])
        age_constraint["$lte"] = min(existing.get("$lte", 99999), age_range[1])
    base_query["age_upon_outcome_in_weeks"] = age_constraint

    # --- Data table: fetch filtered documents from MongoDB ---
    documents = shelter.read(base_query, projection={'_id': 0})
    filtered_df = pd.DataFrame(documents)

    if filtered_df.empty:
        empty_fig = px.scatter(title="No data matching the current filters.")
        return [], empty_fig, empty_fig, empty_fig

    table_data = filtered_df.to_dict('records')

    # --- Pie chart: use aggregation pipeline for outcome distribution ---
    # aggregate_outcomes() runs $match -> $group -> $sort inside MongoDB,
    # returning only the summary rows rather than all raw documents.
    outcome_counts = shelter.aggregate_outcomes(match=base_query)
    if outcome_counts:
        agg_df = pd.DataFrame(outcome_counts)
        pie_fig = px.pie(
            agg_df,
            names='outcome_type',
            values='count',
            title='Outcome Type Distribution (aggregation pipeline)'
        )
    else:
        pie_fig = px.pie(
            filtered_df,
            names='outcome_type',
            title='Outcome Type Distribution'
        )

    # --- Breed bar chart: top 10 breeds in the filtered result ---
    breed_counts = (
        filtered_df['breed']
        .value_counts()
        .head(10)
        .reset_index()
    )
    breed_counts.columns = ['breed', 'count']
    breed_fig = px.bar(
        breed_counts,
        x='breed',
        y='count',
        title='Top 10 Breeds in Current Filter',
        labels={'breed': 'Breed', 'count': 'Number of Animals'}
    )
    breed_fig.update_layout(xaxis_tickangle=-35)

    # --- Geolocation map ---
    geo_df = filtered_df.dropna(subset=['location_lat', 'location_long'])
    geo_fig = px.scatter_mapbox(
        geo_df,
        lat='location_lat',
        lon='location_long',
        color='animal_type',
        hover_name='name',
        hover_data=['breed', 'age_upon_outcome_in_weeks', 'outcome_type'],
        zoom=10,
        height=500
    )
    geo_fig.update_layout(mapbox_style="open-street-map")

    return table_data, pie_fig, breed_fig, geo_fig


In [ ]:
# Cell 6: Run the application
if __name__ == "__main__":
    app.run(port=8050, debug=True)
